# Fig 9 Validation — Flame projection lengths and deflection angles in x-y plane

Reproduces Lan et al. 2023 (Ocean Engineering 281:114890) Fig 9 from our Mesh 5 (1.07 M cells) runs of
Method 1 / 2 / 3 at v = 1 m/s, T_END = 300 s.

**Quantities (HRRPUV ≥ 200 kW/m³ bounding box, per second):**
- `Lx = flame_xmax − flame_xmin` — horizontal flame width in x
- `Ly = flame_ymax − flame_ymin` — horizontal flame width in y
- `θy = arctan(Lx/Ly) × 57.3°` — deflection angle from y-axis

**Expected paper behavior:**
- Method 1 (asymmetric): Lx ≠ 0, Ly ≈ 0.7, θy ≈ 45–55°
- Method 2/3 (symmetric): Lx ≈ 0, Ly ≈ 1.0–1.2 m, θy ≈ 90°

In [ ]:
import subprocess, sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/scratch/x3319a05/fds/Engine_Room')
RESULTS = ROOT / 'Results'
FLAME_DIR = ROOT / 'analysis' / 'flame'
FIG9_CSV  = FLAME_DIR / 'fig9_data.csv'

print('matplotlib', plt.matplotlib.__version__, '| pandas', pd.__version__)
print('Results dir:', RESULTS)
print('Method runs found:', sorted(p.name for p in RESULTS.glob('method*_v*_*')))

## 1. Extract flame DEVCs from Method runs

Runs `extract_flame.py` against `Results/method*_v*_*` directories. If the FDS jobs are still going (PENDING/RUNNING), this cell will be a no-op or partial; just re-run after jobs finish.

In [ ]:
cmd = ['python3', str(FLAME_DIR / 'extract_flame.py'),
       '--results-dir', str(RESULTS),
       '--out-dir',     str(FLAME_DIR)]
p = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:'); print(p.stdout[-2000:])
print('STDERR:'); print(p.stderr[-2000:])
print('exit code:', p.returncode)

In [ ]:
if not FIG9_CSV.exists():
    raise SystemExit(f'NO DATA YET: {FIG9_CSV} not found. Re-run after FDS jobs complete.')
df = pd.read_csv(FIG9_CSV)
print(f'rows: {len(df)},  cols: {list(df.columns)}')
print('methods present:', sorted(df.method.unique()))
print('runs present:   ', sorted(df.run.unique()))
df.head()

## 2. Fig 9 (top) — Method 1 asymmetric: Lx, Ly, θy vs time

In [ ]:
m1 = df[df.method == 1].sort_values('time_s')
if m1.empty:
    print('No Method 1 data yet.')
else:
    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    ax2 = ax1.twinx()
    ax1.plot(m1.time_s, m1.Ly_m, 'k-s', ms=3, lw=1, mfc='none', label='$L_y$ (m)')
    ax1.plot(m1.time_s, m1.Lx_m, 'r-o', ms=3, lw=1, mfc='none', label='$L_x$ (m)')
    ax2.plot(m1.time_s, m1.theta_y_deg, 'b-^', ms=3, lw=1, mfc='none', label=r'$\theta_y$ (degree)')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('$L_x$, $L_y$ (m)')
    ax2.set_ylabel(r'$\theta_y$ (degree)')
    ax1.set_xlim(0, m1.time_s.max())
    ax1.set_ylim(0.4, 1.6)
    ax2.set_ylim(0, 90)
    ax1.set_title('Method 1 (asymmetric):  $v = 1$ m/s')
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1+h2, l1+l2, loc='upper right')
    plt.tight_layout()
    plt.savefig(FLAME_DIR / 'fig9_method1.png', dpi=150)
    plt.show()

## 3. Fig 9 (bottom) — Method 2 & 3 symmetric: Ly vs time

(For symmetric layouts Lx ≈ 0 → θy ≈ 90°, so paper only plots Ly.)

In [ ]:
m23 = df[df.method.isin([2, 3])].sort_values(['method', 'time_s'])
if m23.empty:
    print('No Method 2/3 data yet.')
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for mid, color, marker in [(2, 'r', 'o'), (3, 'b', '^')]:
        sub = m23[m23.method == mid]
        if not sub.empty:
            ax.plot(sub.time_s, sub.Ly_m, color=color, marker=marker,
                    ms=3, lw=1, mfc='none', label=f'Method {mid}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('$L_y$ (m)')
    ax.set_xlim(0, m23.time_s.max())
    ax.set_ylim(0.4, 1.6)
    ax.set_title('Method 2 & 3 (symmetric):  $v = 1$ m/s')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(FLAME_DIR / 'fig9_method23.png', dpi=150)
    plt.show()

## 4. Quantitative summary — steady-state window (t ≥ 100 s)

Compare time-averaged values against paper Fig 9 reads.

In [ ]:
STEADY_START = 100   # s — exclude transient growth
PAPER = {
    1: {'Ly': (0.65, 0.78), 'Lx': (0.65, 0.90), 'theta_y': (45, 55)},
    2: {'Ly': (1.00, 1.20)},
    3: {'Ly': (0.90, 1.10)},
}
rows = []
for mid in sorted(df.method.unique()):
    sub = df[(df.method == mid) & (df.time_s >= STEADY_START)]
    if sub.empty:
        continue
    row = {'method': int(mid)}
    row['Ly_mean']      = float(sub.Ly_m.mean())
    row['Lx_mean']      = float(sub.Lx_m.mean())
    row['theta_y_mean'] = float(sub.theta_y_deg.mean())
    if 'Ly' in PAPER[mid]:
        row['Ly_paper']  = f"{PAPER[mid]['Ly'][0]:.2f}-{PAPER[mid]['Ly'][1]:.2f}"
    if 'Lx' in PAPER[mid]:
        row['Lx_paper']  = f"{PAPER[mid]['Lx'][0]:.2f}-{PAPER[mid]['Lx'][1]:.2f}"
    if 'theta_y' in PAPER[mid]:
        row['theta_y_paper'] = f"{PAPER[mid]['theta_y'][0]}-{PAPER[mid]['theta_y'][1]}°"
    rows.append(row)
summary = pd.DataFrame(rows).round(3)
summary

## 5. Combined comparison plot (paper-style two-panel)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

ax = axes[0]; axR = ax.twinx()
m1 = df[df.method == 1].sort_values('time_s')
if not m1.empty:
    ax.plot(m1.time_s, m1.Ly_m, 'k-s', ms=3, lw=1, mfc='none', label='$L_y$ (m)')
    ax.plot(m1.time_s, m1.Lx_m, 'r-o', ms=3, lw=1, mfc='none', label='$L_x$ (m)')
    axR.plot(m1.time_s, m1.theta_y_deg, 'b-^', ms=3, lw=1, mfc='none', label=r'$\theta_y$ (°)')
ax.set_ylabel('$L_x$, $L_y$ (m)')
axR.set_ylabel(r'$\theta_y$ (°)')
ax.set_ylim(0.4, 1.6); axR.set_ylim(0, 90)
ax.set_title('Method 1: asymmetric ventilation,  $v=1$ m/s')
h1,l1 = ax.get_legend_handles_labels(); h2,l2 = axR.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, loc='upper right')

ax = axes[1]
for mid, color, marker in [(2, 'r', 'o'), (3, 'b', '^')]:
    sub = df[df.method == mid].sort_values('time_s')
    if not sub.empty:
        ax.plot(sub.time_s, sub.Ly_m, color=color, marker=marker, ms=3, lw=1, mfc='none',
                label=f'Method {mid}')
ax.set_xlabel('Time (s)')
ax.set_ylabel('$L_y$ (m)')
ax.set_ylim(0.4, 1.6)
ax.set_title('Method 2 & 3: symmetric ventilation,  $v=1$ m/s')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig(FLAME_DIR / 'fig9_combined.png', dpi=150)
plt.show()

## 6. Job status checker

If results are missing, run this cell to check whether the FDS jobs are still pending/running.

In [ ]:
p = subprocess.run(['squeue', '-u', os.getenv('USER', 'x3319a05'),
                    '-o', '%i %j %T %M %l %D %R'],
                   capture_output=True, text=True)
print(p.stdout)